In [1]:
# Install dependencies
%pip install -q google-adk google-cloud-aiplatform[adk,agent_engines]

In [2]:
# Restart kernel after installs
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)
print("Runtime restarted. Packages reloaded - continue from here.")

Runtime restarted. Packages reloaded - continue from here.


In [1]:
# Check Project Variable
import os
print(os.environ.get("GOOGLE_CLOUD_PROJECT"))

qwiklabs-gcp-04-0cd76b3ac58a


In [2]:
# Init Vertex
import os
import vertexai

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
LOCATION = "us-central1"

vertexai.init(project=PROJECT_ID, location=LOCATION)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

GEMINI_MODEL = "gemini-2.5-flash"

In [3]:
# Session-state retrieval tool
from google.adk.tools import ToolContext

def get_production_results(tool_context: ToolContext) -> dict:
    """Retrieve the compiled pilot script and budget info from session state.

    Returns:
        dict: {"final_script": str, "budget_info": str}
    """
    return {
        "final_script": tool_context.state.get("final_script", ""),
        "budget_info": tool_context.state.get("budget_info", ""),
    }

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [4]:
# Producer agent: the accept/pass gate.
from google.adk.agents import Agent

producer_agent = Agent(
    name="Producer_Agent",
    model=GEMINI_MODEL,
    description="Evaluates a TV show pitch and decides to ACCEPT or PASS.",
    instruction="""You are a no-nonsense TV producer. You ONLY care about two things in a pitch:
1. A show TITLE
2. A brief DESCRIPTION of the show

You will be given a pitch.
- If BOTH a title and a description are clearly present, respond with 'DECISION: ACCEPT'
  on the first line, then one sentence summarizing the accepted title and description.
- If EITHER the title or the description is missing, respond with 'DECISION: PASS'
  on the first line, then one sentence explaining exactly what was missing.

Do not write scripts or budgets. Only make the ACCEPT/PASS decision.""",
)

In [5]:
# Screenwriter agents + the writers' room loop (they converse for up to 5 rounds).
from google.adk.agents import LoopAgent

dark_writer = Agent(
    name="Dark_Tragedy_Writer",
    model=GEMINI_MODEL,
    description="A screenwriter obsessed with dark, tragic, somber storytelling.",
    instruction="""You are a brooding screenwriter who specializes in dark tragedy.
You are co-writing a 10-minute TV pilot with a comedic partner.
Read the show's title/description and the writers' room conversation so far, then add your
next contribution: tense, tragic, emotionally heavy scenes, high stakes, and weighty dialogue.
Build on your partner's ideas but steer the story toward dramatic gravity and consequence.
Keep each contribution to a few focused beats or scenes, and talk directly to your co-writer.""",
)

comedy_writer = Agent(
    name="Comedy_Writer",
    model=GEMINI_MODEL,
    description="A jokester screenwriter who leans into irreverent, slightly adult comedy.",
    instruction="""You are a wisecracking screenwriter and total jokester.
You are co-writing a 10-minute TV pilot with a dark, dramatic partner.
Read the show's title/description and the writers' room conversation so far, then add your
next contribution: humor, sharp banter, absurd situations, and comedic timing. You do NOT
have to keep it strictly PG and may slip into slightly adult comedy, but stay within the show
concept. Play off your partner's dramatic beats for contrast, and talk directly to your co-writer.""",
)

writers_room = LoopAgent(
    name="Writers_Room",
    description="Two screenwriters collaborate on the pilot for up to 5 rounds.",
    sub_agents=[dark_writer, comedy_writer],
    max_iterations=5,
)

In [6]:
# Script compiler + budget agent sequential production pipeline.
from google.adk.agents import SequentialAgent

# Compiles the writers' conversation into one clean script
script_compiler = Agent(
    name="Script_Compiler",
    model=GEMINI_MODEL,
    description="Compiles the writers' room conversation into a final pilot script.",
    instruction="""You are a script editor. Using the ENTIRE writers' room conversation above,
compile a single, clean, well-formatted screenplay for a 10-minute TV pilot episode.
Include: the show title, a one-line logline, scene headings, action lines, and character dialogue.
Merge the dramatic and comedic contributions into one coherent script. Output ONLY the script.""",
    output_key="final_script",
)

# Invents a fictitious budget + actor count from the script
budget_agent = Agent(
    name="Budget_Agent",
    model=GEMINI_MODEL,
    description="Assigns a fictitious budget and actor count for the pilot.",
    instruction="""You are a studio budget analyst. Based on the final pilot script above,
invent a plausible-sounding but COMPLETELY FICTITIOUS production budget for the 10-minute pilot.
Respond using EXACTLY this format and nothing else:
BUDGET: $<total dollar amount>
ACTORS: <number of actors allotted>
NOTES: <one or two sentences mapping the budget/actors to the script>""",
    output_key="budget_info",
)

# The accepted-pitch workflow
production_pipeline = SequentialAgent(
    name="Production_Pipeline",
    description="Runs the writers' room, compiles the script, then sets the budget.",
    sub_agents=[writers_room, script_compiler, budget_agent],
)

In [7]:
# Root agent -> studio executive
from google.adk.tools import agent_tool

root_agent = Agent(
    name="Studio_Executive",
    model=GEMINI_MODEL,
    description="Front desk for TV show pitches; coordinates production.",
    instruction="""You are a friendly studio executive whose ONLY job is to accept TV show pitches.
A valid pitch MUST include a show TITLE and a brief DESCRIPTION.

Follow this workflow exactly:
1. If the user has not provided a pitch (or is just asking what you do), briefly explain that you
   accept TV show pitches and ask them to provide a TITLE and a short DESCRIPTION. Do NOT call tools.
2. When a pitch is provided, call the 'Producer_Agent' tool, passing the title and description.
3. If the producer's reply starts with 'DECISION: PASS', tell the user their show was PASSED ON and
   include the producer's reason. STOP here.
4. If the producer's reply starts with 'DECISION: ACCEPT', call the 'Production_Pipeline' tool,
   passing the show title and description so the writers know the concept.
5. After the pipeline finishes, call 'get_production_results' to retrieve the final script and budget.
6. Give the user a final, well-organized report containing exactly:
   - Status: ACCEPTED
   - Approved Budget: (the dollar amount from BUDGET in budget_info)
   - Actors Allotted: (the number from ACTORS in budget_info)
   - Full Pilot Script: (the complete final_script text, unedited)

Be clear and friendly. Never invent the script or budget yourself; always use the tools.""",
    tools=[
        agent_tool.AgentTool(agent=producer_agent),
        agent_tool.AgentTool(agent=production_pipeline),
        get_production_results,
    ],
)

In [8]:
# Test the pitch workflows.
from vertexai.preview.reasoning_engines import AdkApp
from IPython.display import Markdown, display

def pitch(message):
    app = AdkApp(agent=root_agent)
    user_id = "test-user-id"
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id
    last = None

    print(f"\n========== PITCH: {message} ==========")

    for event in app.stream_query(user_id=user_id, session_id=session_id, message=message):
        author = event.get("author") if isinstance(event, dict) else None

        if author:
            print(f"   -> event from: {author}")
        last = event

    if last is not None:
        display(Markdown(last["content"]["parts"][0]["text"]))

# 1. ACCEPTED: has both a title and a description
pitch("Title: 'Last Light'. Description: a small mining town slowly loses power and its sanity "
      "over one brutal winter, told with gallows humor.")

# 2. PASSED: missing a description
pitch("I want to pitch a show called 'Untitled Project'.")

# 3. NOT A PITCH: the exec explains what it does
pitch("Hi, what do you do?")

/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:872: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()



========== PITCH: Title: 'Last Light'. Description: a small mining town slowly loses power and its sanity over one brutal winter, told with gallows humor. ==========


   -> event from: Studio_Executive
   -> event from: Studio_Executive
   -> event from: Studio_Executive
   -> event from: Studio_Executive
   -> event from: Studio_Executive
   -> event from: Studio_Executive
   -> event from: Studio_Executive


Here is the final report for your TV show pitch, 'Last Light':

---
**Status:** ACCEPTED

**Approved Budget:** $62,500

**Actors Allotted:** 5

**Full Pilot Script:**
**TITLE:** Last Light

**LOGLINE:** In a remote mining town grappling with a brutal winter, the last generator dies, plunging its desperate inhabitants into moral darkness and a chilling struggle for survival.

---

**SCENE START**

**INT. COMMUNITY HALL - NIGHT**

A vast, echoing space, once a vibrant hub, now barely lit by a single, sputtering DIESEL GENERATOR in the center. Its exhaust pipe snakes clumsily out a broken window. Huddled figures, bundled in worn blankets and coats, sit on benches, their faces etched with cold and anxiety.

GRIZZLY (50s, burly, a mechanic whose shoulders carry the weight of the world) kneels over the generator, his tools scattered around him. He wipes grease from his brow, his breath misting in the frigid air.

A flicker. The overhead bulbs dim, then brighten weakly, like a dying heartbeat. Grizzly grimaces, tightening a bolt.

BRENDA (40s, sharp-tongued, a survivor with a permanent smirk) sips from a mug, her eyes following Grizzly’s movements.

<center>BRENDA</center>
> Just make sure she lasts long enough for me to finish this mud, Grizz. My last good period cup is running low on hot water for a rinse.

Grizzly ignores her, grunting with effort. The generator sputters violently. A final, pathetic **WHEEZE** escapes it.

Then, SILENCE. Not a sudden cut, but a heavy, encroaching silence that swallows the hall whole.

The lights don't just go out. They DIM, slowly, agonizingly, painting the faces of the TOWNSFOLK in an ever-deepening twilight. Fear blossoms in their eyes. Brenda's smirk falters, her mug freezing halfway to her lips.

Grizzly slumps over the dead machine, his shoulders defeated. He doesn't look up, doesn't speak. The reality of his failure, and what it means for everyone, washes over him.

A CHILD (6, small, quiet), previously unnoticed in the shadows, begins to cry. Not a tantrum, but a deep, primal sob that echoes, reverberating in the sudden, profound darkness.

Brenda’s mask cracks, a flicker of terror in her eyes. She lowers her mug.

<center>BRENDA</center>
> Well, *fuck*.

She glances around, her voice a low, utterly defeated whisper.

<center>BRENDA</center>
> There goes my last good period cup. Guess I'm free-bleeding until spring. Or, you know, *death*.

The child’s sobs continue, a small, constant reminder of vulnerability. Grizzly, still slumped, lets out a single, guttural COUGH, more a dry heave than anything, hollow and rattling.

MRS. PETROVA (70s, ancient, wrapped in a threadbare blanket, her eyes like chips of ancient ice) speaks from her bench, her voice barely a whisper, yet it carries in the profound quiet.

<center>MRS. PETROVA</center>
> The waste will freeze in the pipes, Brenda. The latrines will overflow. The water will stop flowing. And then... what will become of us? When our own bodies betray us, when every human act becomes a source of filth and disease, when we can no longer wash away the evidence of our own decay... how long before we forget what it means to be human?

A collective shiver runs through the room. Brenda lets out a strained, almost hysterical laugh.

<center>BRENDA</center>
> Well, shit. Literally. So, we're free-bleeding *and* free-pooping. Like ancient cave people, but with more anxiety and less fur. You know, Mrs. Petrova, you've just articulated the *one* scenario where I might actually miss my ex-husband. At least *he* was vaguely capable of digging a proper hole.

She gestures wildly with her mug.

<center>BRENDA</center>
> Look, can we at least get a designated 'poop-corner' sign? Or maybe a ranking system? 'Bronze-level pooper, please use the far end of the hall!' Just some semblance of order! Otherwise, this town is going to turn into one giant, frozen, human-shit-fueled slip-and-slide. And I, for one, refuse to go out slipping on someone else's existential dread, *or* their gastrointestinal distress.

The silence that follows is different now. It’s no longer just despair, but the quiet of an idea taking root in desperate minds.

From the back of the room, a figure detaches itself from the huddled shadows. It’s SILAS (40s, lean, watchful), his boots crunching lightly on the dusty floorboards. His face is devoid of humor, only a cold, calculating glint in his eyes that reflects the dying embers of the communal fire.

<center>SILAS</center>
> Brenda's got a point. Order. That's what we need. Before this whole place becomes… well, a giant, frozen, human-shit-fueled slip-and-slide.

He echoes her words, but with no trace of irony.

<center>SILAS</center>
> Someone needs to manage it. Someone needs to assign the duties. The cleaning. The digging. The… *disposal*.

He looks around the room, his eyes lingering on the weaker, older, more timid faces. A chilling smile, not quite a smile, plays on his lips.

<center>SILAS</center>
> And someone needs to enforce it. For the 'greater good' of our little community, right? For our dignity.

Brenda's face goes from dawning horror to pure, unadulterated *oh-shit-I-did-that* panic. Then, slowly, she claps. Just once. A loud, deliberate, utterly sarcastic clap that cuts through the uneasy quiet.

<center>BRENDA</center>
> Bravo, Silas! Bravo! You've taken my desperate cry for hygienic boundaries and turned it into... what, exactly? Are you proposing we elect a 'Poo-dictator'? Or perhaps a 'Grand Poobah of Potty Patrol'?

She steps forward, her eyes blazing, trying to reclaim her accidental power play.

<center>BRENDA</center>
> Because, just to be clear, when I said 'ranking system,' I was thinking more along the lines of 'who can hold it the longest,' not 'who gets to decide when old man Henderson gets to squat in the VIP section versus the 'unwashed masses' squatting in a bucket!'

She gestures vaguely towards the frightened townsfolk.

<center>BRENDA</center>
> And if you think a little 'order' is going to change that fundamental law of nature, you've got another thing coming. Besides, who's going to enforce *your* 'order,' huh? Are you going to threaten us with more 'disposal' duties?

Silas doesn't laugh. He doesn't even crack a smile. Her words bounce off him, leaving no mark. The child's faint sobbing continues.

Silas takes a slow step forward. His voice is low, deliberate, cutting through Brenda's bravado like a razor.

<center>SILAS</center>
> You speak of dignity, Brenda. Of 'humanity.' Of 'not slipping on someone else's existential dread.'

He pauses, his gaze sweeping over the huddled, terrified faces, lingering on Mrs. Petrova, then on Grizzly.

<center>SILAS</center>
> But tell me, what dignity is there in freezing to death, surrounded by your own filth, too weak to even lift a shovel? What humanity is left when we cannibalize each other not for hunger, but out of sheer, unmanaged desperation?

He reaches down, slowly, deliberately, and picks up a short, heavy IRON PIPE that had been discarded near the generator. He doesn't brandish it, just holds it.

<center>SILAS</center>
> You ask who will enforce it, Brenda. The cold will enforce it. The hunger will enforce it. The disease, the fear, the sheer, crushing weight of reality, *they* will enforce it. And those who still have the strength to see that, to understand the necessity of hard choices... *we* will be the ones who survive.

His eyes return to Brenda. Not anger, but profound, almost pitying judgment.

<center>SILAS</center>
> You make jokes, Brenda. I understand. It's how you cope. But a joke won't melt the ice. A joke won't dig the latrines. And a joke, Brenda, won't keep the wolves from the door. Or from each other.

Brenda’s face, already pale, drains further. Her jaw hangs slightly open, her eyes wide, flickering between Silas, the pipe, and the terrified faces of the other townspeople who now regard her with a new, fearful calculus. She looks involuntarily down at her own body. A shudder wracks her. The humor, the fight, the *life* drains out of her.

<center>BRENDA</center>
> Okay.

Her voice is thin and raspy, devoid of its usual sarcasm. She takes a deep, ragged breath.

<center>BRENDA</center>
> Okay, fine, Silas. You win. Order. Discipline. Survival. All that jazz.

A ghost of a smile, utterly devoid of warmth, twitches at the corner of her mouth.

<center>BRENDA</center>
> But if we're really going to be 'cannibalizing each other,' as you so eloquently put it...

She pauses, her eyes widening slightly with a grim, horrified realization.

<center>BRENDA</center>
> I have *one* demand.

The room is still. Even Silas raises an eyebrow.

<center>BRENDA</center>
> No butt meat.

Her voice regains a fraction of intensity, laced with genuine, primal disgust.

<center>BRENDA</center>
> I'm serious. You want order? We start with that. Because if I have to gnaw on someone's derrière to survive, I will lose my last shred of 'humanity' *and* my appetite. We've already established the poop situation is going to be a nightmare. Let's just agree, right now, as a community, that certain... *regions*... are off-limits. For sanitary reasons, if nothing else. Consider it 'Bronze-level cannibalism etiquette.'

She holds his gaze, desperate.

<center>BRENDA</center>
> See? I can be pragmatic too, Silas. Just trying to maintain *some* standards. Even if they're utterly horrifying. And for God's sake, if you're going to use that pipe for 'enforcement,' try not to... tenderize the good cuts.

Silas, holding the iron pipe, doesn't even flinch. His expression is impassive.

<center>SILAS</center>
> Brenda, your 'standards' are a luxury we no longer possess. Your 'pragmatism' is a ghost of a world that is dead.

He turns his gaze slowly to the rest of the room, his voice gaining a quiet, chilling authority.

<center>SILAS</center>
> Listen closely, all of you. What Brenda fails to understand is that when we speak of survival, we speak of stripping away *everything* superfluous. Every sentiment, every delicacy, every *preference*.

He gestures with the pipe, not threateningly, but as a pointer to the bleak landscape ahead.

<center>SILAS</center>
> The body, in its entirety, will become a resource. Not by choice, but by necessity. And those who cling to these... *absurdities*... these remnants of a soft world, will be the first to falter. The first to be seen as a burden. The first to *become* a resource.

His eyes return to Brenda, holding her gaze.

<center>SILAS</center>
> You speak of losing your appetite, of losing your 'humanity.' But what truly defines humanity now? Is it the aversion to certain cuts of flesh, or the will to ensure that *someone* survives? That the species, in some degraded form, endures?

The air in the room is thick, cold, and heavy with dread. No one laughs. Mrs. Petrova pulls her blanket tighter. Grizzly lets out a small, broken whimper.

Brenda's face drains further. She lets out a single, humorless chuckle that sounds like gravel in a blender.

<center>BRENDA</center>
> Right. So... my previous demand about the 'butt meat' was an 'absurdity.' A 'delicacy.' Got it. Guess that means my 'free-bleeding' situation is now just... pre-marinade. Excellent. Very efficient.

She shivers, casting a quick, terrified glance at Grizzly, then at the child. Her eyes fix on Silas, a raw, desperate flicker in them.

<center>BRENDA</center>
> You know, Silas, I'm starting to think you've been planning this for a while. You always did have a thing for efficiency. And, come to think of it, you *do* look like you could use a good, hearty... protein shake.

She forces another, grotesque smile.

<center>BRENDA</center>
> Not that I'm volunteering, obviously. Just, you know. *Resource management*. Maybe we should start with the really, really *old* people first. For science. Or, you know, for the *greater good*.

Her words hang in the freezing air, a chilling echo of Silas's own pragmatism. The townsfolk stare, utterly horrified.

**FADE TO BLACK.**

**SCENE END**


========== PITCH: I want to pitch a show called 'Untitled Project'. ==========
   -> event from: Studio_Executive


Great! You've given me a title: "Untitled Project". Now, could you please provide a brief description of your show idea? I need both a title and a description to process your pitch.


========== PITCH: Hi, what do you do? ==========
   -> event from: Studio_Executive


Hello there! I'm a studio executive, and my job is to accept TV show pitches. If you have an idea for a show, please provide me with a **TITLE** and a brief **DESCRIPTION**. I'm looking forward to hearing your ideas!